
# Grain Robustness Q1 — Notebook 02
## Redundancia estructural y construcción formal de representaciones

Este notebook utiliza exclusivamente la salida validada del Notebook 01:

`NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip`

### Objetivos

1. Cargar los datasets congelados de Dry Bean e INIAP Rice.
2. Armonizar sus mediciones morfométricas en un esquema común.
3. Diferenciar mediciones primitivas de variables algebraicamente derivadas.
4. Construir representaciones comparables entre cultivos.
5. Verificar matemáticamente la invariancia ante cambio sintético de escala.
6. Cuantificar redundancia, correlación y condicionamiento numérico.
7. Generar los archivos que utilizarán los notebooks de partición y modelado.

### Representaciones principales

- **R1 — Full native:** todas las variables originales disponibles.
- **R2 — Measured core:** seis mediciones geométricas comunes.
- **R3 — Invariant core:** cinco descriptores adimensionales no redundantes por construcción.
- **R4 — Hybrid:** invariantes más una sola ancla de tamaño, `log_area`.

### Importante

- No se entrena ningún clasificador.
- No se crean aún particiones de entrenamiento y prueba.
- No se elimina ningún registro.
- Los duplicados de Dry Bean conservan su `duplicate_group_id`.
- El cambio de escala se aplica sobre variables primitivas y luego se recalculan las derivadas.
- Utilizar **CPU**, sin GPU.


## 1. Preparar Colab

In [ ]:

!pip -q install pyarrow openpyxl


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Importaciones y configuración

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import shutil
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 20260803
SCALE_FACTORS = [0.50, 0.67, 0.80, 1.00, 1.25, 1.50, 2.00]

LOCAL_ROOT = Path('/content/Grain_Robustness_Q1_notebook02')
INPUT_DIR = LOCAL_ROOT / '00_input'
EXTRACT_DIR = LOCAL_ROOT / '01_extracted_notebook01'
OUTPUT_DIR = LOCAL_ROOT / '02_outputs'
DATA_DIR = OUTPUT_DIR / '04_processed_features'
RESULTS_DIR = OUTPUT_DIR / '07_results' / '02_REDUNDANCY_REPRESENTATIONS'
FIGURES_DIR = OUTPUT_DIR / '08_figures' / '02_REDUNDANCY_REPRESENTATIONS'
TABLES_DIR = OUTPUT_DIR / '09_tables' / '02_REDUNDANCY_REPRESENTATIONS'
PROTOCOL_DIR = OUTPUT_DIR / '03_protocol'

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Grain_Robustness_Q1')

for directory in [
    INPUT_DIR,
    EXTRACT_DIR,
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    PROTOCOL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print('Espacio local:', LOCAL_ROOT)
print('Destino en Drive:', DRIVE_PROJECT_ROOT)
print('CPU disponibles:', os.cpu_count())
print('Python:', sys.version.split()[0])


## 3. Localizar el ZIP del Notebook 01

In [ ]:

from google.colab import files

candidate_paths = [
    Path('/content/NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip'),
    DRIVE_PROJECT_ROOT / '07_results' / 'NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip',
    DRIVE_PROJECT_ROOT / 'NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip',
]

zip_source = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if zip_source is None:
    print(
        'No se encontró automáticamente el ZIP. '
        'Selecciona NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip.'
    )
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())

    if len(uploaded_names) != 1:
        raise RuntimeError(
            'Debes cargar exactamente un archivo ZIP.'
        )

    zip_source = Path('/content') / uploaded_names[0]

if zip_source.suffix.lower() != '.zip':
    raise ValueError(f'El archivo no es ZIP: {zip_source}')

zip_local = INPUT_DIR / 'NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip'
shutil.copy2(zip_source, zip_local)

print('ZIP utilizado:', zip_source)
print('Copia local:', zip_local)
print('Tamaño MB:', round(zip_local.stat().st_size / 1_000_000, 3))


## 4. Extraer y validar la entrada

In [ ]:

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_local, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

required_files = {
    'dry_bean': EXTRACT_DIR / '02_data_frozen' / 'dry_bean_frozen_v1.parquet',
    'iniap_rice': EXTRACT_DIR / '02_data_frozen' / 'iniap_rice_frozen_v1.parquet',
    'audit_summary': (
        EXTRACT_DIR / '07_results' / '01_AUDIT_FREEZE'
        / '03_dataset_audit_summary.csv'
    ),
    'freeze_validation': (
        EXTRACT_DIR / '07_results' / '01_AUDIT_FREEZE'
        / '09_freeze_validation.csv'
    ),
    'algebraic_relationships': (
        EXTRACT_DIR / '09_tables' / '01_AUDIT_FREEZE'
        / 'Table_algebraic_relationships.csv'
    ),
}

missing = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        'El ZIP no contiene todos los archivos requeridos:\n'
        + '\n'.join(missing)
    )

freeze_validation = pd.read_csv(
    required_files['freeze_validation']
)

if not freeze_validation.drop(
    columns=['dataset_id']
).all(axis=None):
    raise RuntimeError(
        'La validación del Notebook 01 no fue completamente satisfactoria.'
    )

print('Entrada del Notebook 01 validada correctamente.')
display(freeze_validation)


## 5. Cargar los datasets congelados

In [ ]:

datasets = {
    'dry_bean': pd.read_parquet(required_files['dry_bean']),
    'iniap_rice': pd.read_parquet(required_files['iniap_rice']),
}

expected_rows = {
    'dry_bean': 13611,
    'iniap_rice': 9200,
}

for dataset_id, dataframe in datasets.items():
    if len(dataframe) != expected_rows[dataset_id]:
        raise RuntimeError(
            f'Tamaño inesperado para {dataset_id}: {len(dataframe)}'
        )

    if not dataframe['record_id'].is_unique:
        raise RuntimeError(
            f'Los record_id no son únicos en {dataset_id}.'
        )

    if dataframe['duplicate_group_id'].isna().any():
        raise RuntimeError(
            f'Hay duplicate_group_id ausentes en {dataset_id}.'
        )

    print(dataset_id, dataframe.shape)
    display(dataframe.head(3))



## 6. Armonizar mediciones primitivas

Las seis mediciones comunes son:

1. área;
2. perímetro;
3. eje mayor;
4. eje menor;
5. área convexa;
6. extensión.

Estas columnas se interpretan como mediciones en unidades de imagen. No se
asume ninguna conversión a milímetros.


In [ ]:

COMMON_COLUMN_MAP = {
    'dry_bean': {
        'area': 'Area',
        'perimeter': 'Perimeter',
        'major_axis': 'MajorAxisLength',
        'minor_axis': 'MinorAxisLength',
        'convex_area': 'ConvexArea',
        'extent': 'Extent',
        'label': 'Class',
    },
    'iniap_rice': {
        'area': 'AREA',
        'perimeter': 'PERIMETER',
        'major_axis': 'MAJOR_AXIS',
        'minor_axis': 'MINOR_AXIS',
        'convex_area': 'CONVEX_AREA',
        'extent': 'EXTENT',
        'label': 'CLASS',
    },
}


def build_common_primitive_table(dataset_id, dataframe):
    mapping = COMMON_COLUMN_MAP[dataset_id]

    output = dataframe[
        [
            'dataset_id',
            'record_id',
            'source_excel_row',
            'feature_fingerprint_sha256',
            'full_row_fingerprint_sha256',
            'duplicate_group_id',
            'exact_duplicate_count',
        ]
    ].copy()

    for common_name in [
        'area',
        'perimeter',
        'major_axis',
        'minor_axis',
        'convex_area',
        'extent',
    ]:
        output[common_name] = dataframe[
            mapping[common_name]
        ].astype(float)

    output['class_label'] = (
        dataframe[mapping['label']]
        .astype(str)
    )

    return output


common_primitives = {
    dataset_id: build_common_primitive_table(
        dataset_id,
        dataframe,
    )
    for dataset_id, dataframe in datasets.items()
}

for dataset_id, dataframe in common_primitives.items():
    print(dataset_id, dataframe.shape)
    display(dataframe.head(3))


## 7. Construir variables derivadas con fórmulas explícitas

In [ ]:

def derive_morphometric_features(primitive_dataframe):
    output = primitive_dataframe.copy()

    area = output['area'].to_numpy(dtype=float)
    perimeter = output['perimeter'].to_numpy(dtype=float)
    major = output['major_axis'].to_numpy(dtype=float)
    minor = output['minor_axis'].to_numpy(dtype=float)
    convex_area = output['convex_area'].to_numpy(dtype=float)

    if np.any(area <= 0):
        raise ValueError('El área debe ser positiva.')
    if np.any(perimeter <= 0):
        raise ValueError('El perímetro debe ser positivo.')
    if np.any(major <= 0) or np.any(minor <= 0):
        raise ValueError('Los ejes deben ser positivos.')
    if np.any(convex_area <= 0):
        raise ValueError('El área convexa debe ser positiva.')

    output['aspect_ratio'] = major / minor
    output['eccentricity_reconstructed'] = np.sqrt(
        np.maximum(
            0.0,
            1.0 - (minor / major) ** 2,
        )
    )
    output['solidity'] = area / convex_area
    output['circularity'] = (
        4.0 * np.pi * area / (perimeter ** 2)
    )
    output['ellipse_fill'] = (
        4.0 * area / (np.pi * major * minor)
    )
    output['equivalent_diameter'] = np.sqrt(
        4.0 * area / np.pi
    )
    output['compactness'] = (
        output['equivalent_diameter'] / major
    )
    output['log_area'] = np.log(area)

    return output


engineered = {
    dataset_id: derive_morphometric_features(
        primitive_dataframe
    )
    for dataset_id, primitive_dataframe in common_primitives.items()
}

for dataset_id, dataframe in engineered.items():
    print(dataset_id)
    display(
        dataframe[
            [
                'area',
                'perimeter',
                'major_axis',
                'minor_axis',
                'aspect_ratio',
                'solidity',
                'circularity',
                'ellipse_fill',
                'log_area',
            ]
        ].head(3)
    )


## 8. Verificar las fórmulas contra columnas originales

In [ ]:

verification_rows = []


def compare_columns(
    dataset_id,
    relationship,
    observed,
    reconstructed,
    atol,
    rtol,
):
    observed = np.asarray(observed, dtype=float)
    reconstructed = np.asarray(reconstructed, dtype=float)

    difference = np.abs(observed - reconstructed)
    within = np.isclose(
        observed,
        reconstructed,
        atol=atol,
        rtol=rtol,
    )

    verification_rows.append({
        'dataset_id': dataset_id,
        'relationship': relationship,
        'n': len(observed),
        'max_absolute_error': float(difference.max()),
        'mean_absolute_error': float(difference.mean()),
        'proportion_within_tolerance': float(within.mean()),
        'atol': atol,
        'rtol': rtol,
        'passes_all_rows': bool(within.all()),
    })


bean_original = datasets['dry_bean']
bean_engineered = engineered['dry_bean']

compare_columns(
    'dry_bean',
    'AspectRation vs reconstructed aspect_ratio',
    bean_original['AspectRation'],
    bean_engineered['aspect_ratio'],
    1e-12,
    1e-12,
)
compare_columns(
    'dry_bean',
    'Eccentricity vs reconstructed eccentricity',
    bean_original['Eccentricity'],
    bean_engineered['eccentricity_reconstructed'],
    1e-12,
    1e-12,
)
compare_columns(
    'dry_bean',
    'Solidity vs reconstructed solidity',
    bean_original['Solidity'],
    bean_engineered['solidity'],
    1e-12,
    1e-12,
)
compare_columns(
    'dry_bean',
    'roundness vs reconstructed circularity',
    bean_original['roundness'],
    bean_engineered['circularity'],
    1e-12,
    1e-12,
)
compare_columns(
    'dry_bean',
    'ShapeFactor4 vs reconstructed ellipse_fill',
    bean_original['ShapeFactor4'],
    bean_engineered['ellipse_fill'],
    1e-12,
    1e-12,
)
compare_columns(
    'dry_bean',
    'EquivDiameter vs reconstructed equivalent_diameter',
    bean_original['EquivDiameter'],
    bean_engineered['equivalent_diameter'],
    1e-10,
    1e-12,
)
compare_columns(
    'dry_bean',
    'Compactness vs reconstructed compactness',
    bean_original['Compactness'],
    bean_engineered['compactness'],
    1e-12,
    1e-12,
)

rice_original = datasets['iniap_rice']
rice_engineered = engineered['iniap_rice']

compare_columns(
    'iniap_rice',
    'ECCENTRICITY vs reconstructed eccentricity',
    rice_original['ECCENTRICITY'],
    rice_engineered['eccentricity_reconstructed'],
    1e-4,
    1e-4,
)

formula_verification = pd.DataFrame(verification_rows)
formula_verification.to_csv(
    TABLES_DIR / 'Table_formula_verification.csv',
    index=False,
    encoding='utf-8-sig',
)

display(formula_verification)


## 9. Definir las cuatro representaciones principales

In [ ]:

MEASURED_CORE = [
    'area',
    'perimeter',
    'major_axis',
    'minor_axis',
    'convex_area',
    'extent',
]

# Eccentricidad y compacidad no se incluyen simultáneamente con aspect_ratio,
# porque son transformaciones determinísticas de los mismos ejes.
INVARIANT_CORE = [
    'aspect_ratio',
    'extent',
    'solidity',
    'circularity',
    'ellipse_fill',
]

HYBRID = INVARIANT_CORE + ['log_area']

FULL_NATIVE_COLUMNS = {
    'dry_bean': [
        'Area',
        'Perimeter',
        'MajorAxisLength',
        'MinorAxisLength',
        'AspectRation',
        'Eccentricity',
        'ConvexArea',
        'EquivDiameter',
        'Extent',
        'Solidity',
        'roundness',
        'Compactness',
        'ShapeFactor1',
        'ShapeFactor2',
        'ShapeFactor3',
        'ShapeFactor4',
    ],
    'iniap_rice': [
        'AREA',
        'PERIMETER',
        'MAJOR_AXIS',
        'MINOR_AXIS',
        'ECCENTRICITY',
        'CONVEX_AREA',
        'EXTENT',
    ],
}

representation_catalog_rows = []

for dataset_id in datasets:
    definitions = {
        'R1_full_native': FULL_NATIVE_COLUMNS[dataset_id],
        'R2_measured_core': MEASURED_CORE,
        'R3_invariant_core': INVARIANT_CORE,
        'R4_hybrid_log_area': HYBRID,
    }

    for representation_name, features in definitions.items():
        representation_catalog_rows.append({
            'dataset_id': dataset_id,
            'representation': representation_name,
            'n_features': len(features),
            'features': ' | '.join(features),
            'primary_status': (
                'primary'
                if representation_name in [
                    'R2_measured_core',
                    'R3_invariant_core',
                    'R4_hybrid_log_area',
                ]
                else 'reference'
            ),
        })

representation_catalog = pd.DataFrame(
    representation_catalog_rows
)

representation_catalog.to_csv(
    PROTOCOL_DIR / 'representation_catalog.csv',
    index=False,
    encoding='utf-8-sig',
)

display(representation_catalog)


## 10. Guardar tablas de representación

In [ ]:

METADATA_COLUMNS = [
    'dataset_id',
    'record_id',
    'source_excel_row',
    'feature_fingerprint_sha256',
    'full_row_fingerprint_sha256',
    'duplicate_group_id',
    'exact_duplicate_count',
    'class_label',
]

representation_manifest_rows = []


def save_representation(
    dataset_id,
    representation_name,
    dataframe,
    feature_columns,
):
    output = dataframe[
        METADATA_COLUMNS + feature_columns
    ].copy()

    dataset_output_dir = DATA_DIR / dataset_id
    dataset_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    csv_path = (
        dataset_output_dir
        / f'{representation_name}.csv'
    )
    parquet_path = (
        dataset_output_dir
        / f'{representation_name}.parquet'
    )

    output.to_csv(
        csv_path,
        index=False,
        encoding='utf-8-sig',
    )
    output.to_parquet(
        parquet_path,
        index=False,
    )

    for path, file_format in [
        (csv_path, 'csv'),
        (parquet_path, 'parquet'),
    ]:
        representation_manifest_rows.append({
            'dataset_id': dataset_id,
            'representation': representation_name,
            'format': file_format,
            'path': str(path),
            'rows': len(output),
            'feature_count': len(feature_columns),
            'size_bytes': path.stat().st_size,
        })


for dataset_id in datasets:
    # R1 utiliza las columnas originales, pero conserva los metadatos.
    native = datasets[dataset_id].copy()

    native_label = (
        'Class'
        if dataset_id == 'dry_bean'
        else 'CLASS'
    )

    native_output = native[
        [
            'dataset_id',
            'record_id',
            'source_excel_row',
            'feature_fingerprint_sha256',
            'full_row_fingerprint_sha256',
            'duplicate_group_id',
            'exact_duplicate_count',
        ]
        + FULL_NATIVE_COLUMNS[dataset_id]
        + [native_label]
    ].copy()

    native_output = native_output.rename(
        columns={native_label: 'class_label'}
    )

    native_dir = DATA_DIR / dataset_id
    native_dir.mkdir(parents=True, exist_ok=True)

    for suffix, writer in [
        ('csv', lambda path: native_output.to_csv(
            path,
            index=False,
            encoding='utf-8-sig',
        )),
        ('parquet', lambda path: native_output.to_parquet(
            path,
            index=False,
        )),
    ]:
        native_path = native_dir / f'R1_full_native.{suffix}'
        writer(native_path)
        representation_manifest_rows.append({
            'dataset_id': dataset_id,
            'representation': 'R1_full_native',
            'format': suffix,
            'path': str(native_path),
            'rows': len(native_output),
            'feature_count': len(FULL_NATIVE_COLUMNS[dataset_id]),
            'size_bytes': native_path.stat().st_size,
        })

    save_representation(
        dataset_id,
        'R2_measured_core',
        engineered[dataset_id],
        MEASURED_CORE,
    )
    save_representation(
        dataset_id,
        'R3_invariant_core',
        engineered[dataset_id],
        INVARIANT_CORE,
    )
    save_representation(
        dataset_id,
        'R4_hybrid_log_area',
        engineered[dataset_id],
        HYBRID,
    )

representation_manifest = pd.DataFrame(
    representation_manifest_rows
)

representation_manifest.to_csv(
    RESULTS_DIR / '01_representation_file_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(representation_manifest)


## 11. Transformación sintética de escala físicamente coherente

In [ ]:

def apply_synthetic_scale(
    primitive_dataframe,
    scale_factor,
):
    if scale_factor <= 0:
        raise ValueError(
            'El factor de escala debe ser positivo.'
        )

    scaled = primitive_dataframe.copy()

    for length_column in [
        'perimeter',
        'major_axis',
        'minor_axis',
    ]:
        scaled[length_column] = (
            scaled[length_column] * scale_factor
        )

    for area_column in [
        'area',
        'convex_area',
    ]:
        scaled[area_column] = (
            scaled[area_column] * (scale_factor ** 2)
        )

    # Extent se mantiene: es una razón de áreas.
    return derive_morphometric_features(scaled)


scale_audit_rows = []

for dataset_id, baseline in engineered.items():
    baseline_invariants = baseline[
        INVARIANT_CORE
    ].to_numpy(dtype=float)

    for scale_factor in SCALE_FACTORS:
        scaled = apply_synthetic_scale(
            common_primitives[dataset_id],
            scale_factor,
        )

        scaled_invariants = scaled[
            INVARIANT_CORE
        ].to_numpy(dtype=float)

        maximum_absolute_change = float(
            np.max(
                np.abs(
                    scaled_invariants
                    - baseline_invariants
                )
            )
        )

        expected_log_area_shift = (
            2.0 * np.log(scale_factor)
        )

        observed_log_area_shift = (
            scaled['log_area']
            - baseline['log_area']
        )

        scale_audit_rows.append({
            'dataset_id': dataset_id,
            'scale_factor': scale_factor,
            'max_absolute_change_in_invariant_core': (
                maximum_absolute_change
            ),
            'mean_log_area_shift': float(
                observed_log_area_shift.mean()
            ),
            'expected_log_area_shift': float(
                expected_log_area_shift
            ),
            'max_log_area_shift_error': float(
                np.max(
                    np.abs(
                        observed_log_area_shift
                        - expected_log_area_shift
                    )
                )
            ),
            'invariance_passes': (
                maximum_absolute_change < 1e-10
            ),
        })

scale_invariance_audit = pd.DataFrame(
    scale_audit_rows
)

scale_invariance_audit.to_csv(
    TABLES_DIR / 'Table_scale_invariance_audit.csv',
    index=False,
    encoding='utf-8-sig',
)

if not scale_invariance_audit[
    'invariance_passes'
].all():
    raise RuntimeError(
        'La representación invariante cambió de forma inesperada.'
    )

display(scale_invariance_audit)


## 12. Cuantificar redundancia de cada representación

In [ ]:

def representation_diagnostics(dataframe, features):
    matrix = dataframe[features].to_numpy(dtype=float)
    standardized = StandardScaler().fit_transform(matrix)

    correlation = np.corrcoef(
        standardized,
        rowvar=False,
    )

    if len(features) == 1:
        max_abs_correlation = 0.0
    else:
        upper = np.abs(
            correlation[
                np.triu_indices_from(
                    correlation,
                    k=1,
                )
            ]
        )
        max_abs_correlation = float(upper.max())

    singular_values = np.linalg.svd(
        standardized,
        full_matrices=False,
        compute_uv=False,
    )

    numerical_rank = int(
        np.linalg.matrix_rank(standardized)
    )

    positive_singular_values = singular_values[
        singular_values > (
            np.finfo(float).eps
            * max(standardized.shape)
            * singular_values[0]
        )
    ]

    condition_number = float(
        positive_singular_values[0]
        / positive_singular_values[-1]
    )

    pca = PCA(random_state=RANDOM_SEED)
    pca.fit(standardized)
    cumulative = np.cumsum(
        pca.explained_variance_ratio_
    )

    return {
        'n_features': len(features),
        'numerical_rank': numerical_rank,
        'rank_deficiency': len(features) - numerical_rank,
        'condition_number': condition_number,
        'max_absolute_pairwise_correlation': max_abs_correlation,
        'components_for_90_percent': int(
            np.searchsorted(cumulative, 0.90) + 1
        ),
        'components_for_95_percent': int(
            np.searchsorted(cumulative, 0.95) + 1
        ),
        'components_for_99_percent': int(
            np.searchsorted(cumulative, 0.99) + 1
        ),
    }


diagnostic_rows = []

for dataset_id in datasets:
    representation_features = {
        'R1_full_native': FULL_NATIVE_COLUMNS[dataset_id],
        'R2_measured_core': MEASURED_CORE,
        'R3_invariant_core': INVARIANT_CORE,
        'R4_hybrid_log_area': HYBRID,
    }

    for representation_name, features in representation_features.items():
        if representation_name == 'R1_full_native':
            diagnostic_dataframe = datasets[dataset_id]
        else:
            diagnostic_dataframe = engineered[dataset_id]

        diagnostics = representation_diagnostics(
            diagnostic_dataframe,
            features,
        )
        diagnostics.update({
            'dataset_id': dataset_id,
            'representation': representation_name,
        })
        diagnostic_rows.append(diagnostics)

representation_diagnostics_table = pd.DataFrame(
    diagnostic_rows
)

representation_diagnostics_table.to_csv(
    TABLES_DIR / 'Table_representation_diagnostics.csv',
    index=False,
    encoding='utf-8-sig',
)

display(
    representation_diagnostics_table.sort_values(
        ['dataset_id', 'representation']
    )
)


## 13. Matrices de correlación por representación

In [ ]:

for dataset_id in datasets:
    representation_features = {
        'R2_measured_core': MEASURED_CORE,
        'R3_invariant_core': INVARIANT_CORE,
        'R4_hybrid_log_area': HYBRID,
    }

    for representation_name, features in representation_features.items():
        correlation = (
            engineered[dataset_id][features]
            .corr()
        )

        correlation.to_csv(
            RESULTS_DIR
            / f'correlation_{dataset_id}_{representation_name}.csv',
            encoding='utf-8-sig',
        )

        figure, axis = plt.subplots(
            figsize=(8, 7)
        )

        image = axis.imshow(
            correlation.to_numpy(),
            vmin=-1.0,
            vmax=1.0,
            aspect='auto',
        )

        axis.set_xticks(
            np.arange(len(features))
        )
        axis.set_xticklabels(
            features,
            rotation=45,
            ha='right',
        )
        axis.set_yticks(
            np.arange(len(features))
        )
        axis.set_yticklabels(features)
        axis.set_title(
            f'{dataset_id} — {representation_name}'
        )

        for row_index in range(len(features)):
            for column_index in range(len(features)):
                value = correlation.iloc[
                    row_index,
                    column_index,
                ]
                axis.text(
                    column_index,
                    row_index,
                    f'{value:.2f}',
                    ha='center',
                    va='center',
                    fontsize=8,
                )

        figure.colorbar(
            image,
            ax=axis,
            label='Correlación de Pearson',
        )
        figure.tight_layout()

        stem = (
            f'correlation_{dataset_id}_'
            f'{representation_name}'
        )

        figure.savefig(
            FIGURES_DIR / f'{stem}.png',
            dpi=300,
            bbox_inches='tight',
        )
        figure.savefig(
            FIGURES_DIR / f'{stem}.pdf',
            bbox_inches='tight',
        )
        plt.show()


## 14. Figura conceptual de procedencia de variables

In [ ]:

provenance_rows = [
    {
        'feature': 'area',
        'type': 'primitive',
        'formula_or_source': 'measured in pixels squared',
        'scale_behavior': 's^2',
    },
    {
        'feature': 'perimeter',
        'type': 'primitive',
        'formula_or_source': 'measured in pixels',
        'scale_behavior': 's',
    },
    {
        'feature': 'major_axis',
        'type': 'primitive',
        'formula_or_source': 'measured in pixels',
        'scale_behavior': 's',
    },
    {
        'feature': 'minor_axis',
        'type': 'primitive',
        'formula_or_source': 'measured in pixels',
        'scale_behavior': 's',
    },
    {
        'feature': 'convex_area',
        'type': 'primitive',
        'formula_or_source': 'measured in pixels squared',
        'scale_behavior': 's^2',
    },
    {
        'feature': 'extent',
        'type': 'dimensionless',
        'formula_or_source': 'provided ratio',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'aspect_ratio',
        'type': 'derived dimensionless',
        'formula_or_source': 'major_axis / minor_axis',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'eccentricity_reconstructed',
        'type': 'derived dimensionless',
        'formula_or_source': 'sqrt(1 - (minor_axis / major_axis)^2)',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'solidity',
        'type': 'derived dimensionless',
        'formula_or_source': 'area / convex_area',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'circularity',
        'type': 'derived dimensionless',
        'formula_or_source': '4*pi*area / perimeter^2',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'ellipse_fill',
        'type': 'derived dimensionless',
        'formula_or_source': '4*area / (pi*major_axis*minor_axis)',
        'scale_behavior': 'invariant',
    },
    {
        'feature': 'log_area',
        'type': 'derived size anchor',
        'formula_or_source': 'log(area)',
        'scale_behavior': 'log_area + 2*log(s)',
    },
]

feature_provenance = pd.DataFrame(
    provenance_rows
)

feature_provenance.to_csv(
    PROTOCOL_DIR / 'feature_provenance_and_scale_behavior.csv',
    index=False,
    encoding='utf-8-sig',
)

display(feature_provenance)


## 15. Congelar el protocolo del Notebook 02

In [ ]:

protocol = {
    'protocol_name': 'Grain_Robustness_Q1_representation_protocol_v1',
    'created_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'random_seed': RANDOM_SEED,
    'datasets': {
        'dry_bean': {
            'rows': len(datasets['dry_bean']),
            'classes': int(
                datasets['dry_bean']['Class'].nunique()
            ),
            'duplicate_grouping_required': True,
        },
        'iniap_rice': {
            'rows': len(datasets['iniap_rice']),
            'classes': int(
                datasets['iniap_rice']['CLASS'].nunique()
            ),
            'duplicate_grouping_required': True,
            'note': (
                'No exact duplicates were detected, but the same '
                'group-aware split mechanism will be used.'
            ),
        },
    },
    'representations': {
        'R1_full_native': {
            'role': 'reference',
            'description': (
                'All native numeric predictors available in each dataset.'
            ),
        },
        'R2_measured_core': {
            'role': 'primary',
            'features': MEASURED_CORE,
        },
        'R3_invariant_core': {
            'role': 'primary',
            'features': INVARIANT_CORE,
            'note': (
                'Eccentricity and compactness are omitted because they '
                'are deterministic transformations of included geometry.'
            ),
        },
        'R4_hybrid_log_area': {
            'role': 'primary',
            'features': HYBRID,
            'size_anchor': 'log_area',
        },
    },
    'synthetic_scale_factors': SCALE_FACTORS,
    'scaling_rules': {
        'area': 'multiply by s^2',
        'convex_area': 'multiply by s^2',
        'perimeter': 'multiply by s',
        'major_axis': 'multiply by s',
        'minor_axis': 'multiply by s',
        'extent': 'unchanged',
        'derived_features': (
            'recalculate from scaled primitive measurements'
        ),
    },
    'restrictions': [
        'No claim of real multi-camera validation.',
        'No conversion from pixels to physical units.',
        'No model training in Notebook 02.',
        'No test-set inspection before the frozen split notebook.',
    ],
}

protocol_path = (
    PROTOCOL_DIR
    / 'representation_protocol_v1.json'
)

protocol_path.write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    )
)


## 16. Validación final de archivos

In [ ]:

validation_rows = []

for dataset_id in datasets:
    for representation_name in [
        'R1_full_native',
        'R2_measured_core',
        'R3_invariant_core',
        'R4_hybrid_log_area',
    ]:
        path = (
            DATA_DIR
            / dataset_id
            / f'{representation_name}.parquet'
        )
        dataframe = pd.read_parquet(path)

        feature_count = int(
            representation_catalog.loc[
                (
                    representation_catalog['dataset_id']
                    == dataset_id
                )
                & (
                    representation_catalog['representation']
                    == representation_name
                ),
                'n_features',
            ].iloc[0]
        )

        validation_rows.append({
            'dataset_id': dataset_id,
            'representation': representation_name,
            'file_exists': path.exists(),
            'rows_ok': (
                len(dataframe)
                == expected_rows[dataset_id]
            ),
            'record_ids_unique': dataframe['record_id'].is_unique,
            'group_ids_complete': (
                dataframe['duplicate_group_id']
                .notna()
                .all()
            ),
            'class_labels_complete': (
                dataframe['class_label']
                .notna()
                .all()
            ),
            'feature_count_ok': (
                dataframe.shape[1]
                == len(METADATA_COLUMNS)
                + feature_count
            ),
            'missing_feature_cells': int(
                dataframe.drop(
                    columns=METADATA_COLUMNS
                ).isna().sum().sum()
            ),
        })

final_validation = pd.DataFrame(
    validation_rows
)

final_validation.to_csv(
    RESULTS_DIR / '02_final_representation_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

boolean_columns = [
    'file_exists',
    'rows_ok',
    'record_ids_unique',
    'group_ids_complete',
    'class_labels_complete',
    'feature_count_ok',
]

if not final_validation[
    boolean_columns
].all(axis=None):
    raise RuntimeError(
        'Falló una comprobación de las representaciones.'
    )

if final_validation[
    'missing_feature_cells'
].sum() != 0:
    raise RuntimeError(
        'Se detectaron valores faltantes en las representaciones.'
    )

print('Validación final de representaciones: CORRECTA')
display(final_validation)


## 17. Crear ZIP y sincronizar con Google Drive

In [ ]:

zip_local = (
    LOCAL_ROOT
    / 'NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'
)

if zip_local.exists():
    zip_local.unlink()

with zipfile.ZipFile(
    zip_local,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )


def copy_with_retry(source, destination, attempts=8):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            shutil.copy2(source, destination)

            if (
                not destination.exists()
                or destination.stat().st_size
                != source.stat().st_size
            ):
                raise IOError(
                    'La copia no pudo verificarse.'
                )

            return True

        except (
            ConnectionAbortedError,
            ConnectionResetError,
            OSError,
            IOError,
        ) as error:
            last_error = error
            print(
                f'Intento {attempt}/{attempts}: '
                f'{type(error).__name__}: {error}'
            )
            time.sleep(5)

    print(
        'No se pudo copiar a Drive. '
        f'Último error: {last_error}'
    )
    return False


zip_drive = (
    DRIVE_PROJECT_ROOT
    / '07_results'
    / 'NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'
)

copied = copy_with_retry(
    zip_local,
    zip_drive,
)

print('Notebook 02 completado correctamente.')
print('ZIP local:', zip_local)
print('ZIP en Drive:', zip_drive)
print('Copia en Drive verificada:', copied)

if not copied:
    print(
        'Descarga manualmente el ZIP local desde el panel Archivos de Colab.'
    )



## Archivo que debes compartir

Al finalizar, envía:

```text
/content/drive/MyDrive/Grain_Robustness_Q1/07_results/
NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip
```

Si Google Drive falla, descarga la copia local:

```text
/content/Grain_Robustness_Q1_notebook02/
NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip
```

El notebook debe mostrar:

```text
Entrada del Notebook 01 validada correctamente.
Validación final de representaciones: CORRECTA
Notebook 02 completado correctamente.
```
